# 01 — Descriptive Statistics, Normality Tests & Correlation Analysis

**Study:** Artificial Linguistic Veneer (ALV), AI Agency (AIAS), and Oral Defense Performance (ODP)

## Purpose
This notebook produces all analyses required for the *Results — Descriptive Statistics and Preliminary Analyses* section of an APA-style manuscript:

1. **Data loading and quality checks** (missingness, ranges, duplicates).
2. **Descriptive statistics** (*M*, *SD*, *Mdn*, skewness, kurtosis) in APA format.
3. **Normality assessment** — Shapiro–Wilk tests and Q–Q plots.
4. **Pearson correlation matrix** with significance stars and a heatmap visualization.

> **APA note:** Report descriptive statistics as *M* (*SD*); report correlations as *r* (*df* = *N* − 2), with *p* values and 95% CIs where appropriate.

In [ ]:
# ============================================================
# Setup: imports and notebook-wide plotting style
# ============================================================
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)
plt.rcParams["figure.dpi"] = 110
RNG = np.random.default_rng(42)

DATA_PATH = "/mnt/data/raw_data_template (1).csv"
N_VARS = ["ALV", "AI_Agency", "ODP"]
VAR_LABELS = {
    "ALV": "Artificial Linguistic Veneer",
    "AI_Agency": "AI Agency",
    "ODP": "Oral Defense Performance",
}
print("Environment ready.")

In [ ]:
# ============================================================
# 1. Load data and inspect structure
# ============================================================
df = pd.read_csv(DATA_PATH)
print(f"N rows = {len(df)}, columns = {df.columns.tolist()}")
display(df.head())

# Missingness summary
print("\nMissing values per variable:")
print(df[N_VARS].isna().sum())
print(f"\nDuplicate participant IDs: {df['participant_id'].duplicated().sum()}")

In [ ]:
# ------------------------------------------------------------
# Fallback: if the raw template contains placeholders (no observed
# values), synthesize a demonstration dataset consistent with the
# published correlation pattern (r_ALV,AI = -.28; r_ALV,ODP = -.54;
# r_AI,ODP = .46). Replace this cell with real data when available.
# ------------------------------------------------------------
if df[N_VARS].isna().all().all():
    N = 60
    z = RNG.multivariate_normal([0, 0, 0], np.array([
        [1.0, -0.28, -0.54],
        [-0.28, 1.0, 0.46],
        [-0.54, 0.46, 1.0]]), size=N)
    def to_scale(x, mean, sd, lo, hi):
        return np.clip(mean + sd * stats.zscore(x), lo, hi)
    df["ALV"] = to_scale(z[:, 0], 3.0, 0.7, 1, 5)
    df["AI_Agency"] = to_scale(z[:, 1], 4.1, 0.6, 1, 5)
    df["ODP"] = to_scale(z[:, 2], 2.95, 0.75, 1, 5)
    print("NOTE: demonstration data generated (raw file was a blank template). N =", N)
else:
    df = df.dropna(subset=N_VARS).reset_index(drop=True)
    print(f"Observed data used. Final N = {len(df)}")

In [ ]:
# ============================================================
# 2. Descriptive statistics (APA format)
# ============================================================
def describe_apa(s):
    return pd.Series({
        "N": s.notna().sum(),
        "M": s.mean(), "SD": s.std(ddof=1),
        "Mdn": s.median(), "Min": s.min(), "Max": s.max(),
        "Skew": stats.skew(s, bias=False),
        "Kurtosis": stats.kurtosis(s, bias=False),
    })

desc = df[N_VARS].apply(describe_apa).T.rename(index=VAR_LABELS)
display(desc.round(3))

print("\nAPA-style summary (M and SD in parentheses):")
for v in N_VARS:
    print(f"  {VAR_LABELS[v]}: M = {df[v].mean():.2f} (SD = {df[v].std(ddof=1):.2f})")

In [ ]:
# ============================================================
# 3. Normality tests: Shapiro-Wilk + Q-Q plots
# ============================================================
print("Shapiro-Wilk tests for normality")
print("-" * 55)
for v in N_VARS:
    W, p = stats.shapiro(df[v].dropna())
    verdict = "normality not rejected" if p > .05 else "deviates from normality"
    print(f"  {VAR_LABELS[v]:<32} W = {W:.3f}, p = {p:.3f}  ({verdict})")

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, v in zip(axes, N_VARS):
    stats.probplot(df[v].dropna(), dist="norm", plot=ax)
    ax.set_title(f"Q-Q plot: {VAR_LABELS[v]}", fontsize=11)
    ax.get_lines()[0].set(marker="o", markersize=4, color="#2166ac")
    ax.get_lines()[1].set(color="crimson", lw=1.5)
plt.suptitle("Normal Q-Q Plots", y=1.02, fontweight="bold")
plt.tight_layout()
plt.savefig("/mnt/data/qq_plots.png", bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# 4. Correlation matrix with significance stars
# ============================================================
def star(p):
    return "***" if p < .001 else "**" if p < .01 else "*" if p < .05 else ""

k = len(N_VARS)
r_mat = np.zeros((k, k)); p_mat = np.zeros((k, k))
for i, a in enumerate(N_VARS):
    for j, b in enumerate(N_VARS):
        r, p = stats.pearsonr(df[a], df[b])
        r_mat[i, j], p_mat[i, j] = r, p

labels = [VAR_LABELS[v] for v in N_VARS]
corr_df = pd.DataFrame(r_mat, index=labels, columns=labels)

print("Pearson correlations with two-tailed significance (APA table style)")
print("-" * 90)
for i, a in enumerate(N_VARS):
    row = f"  {i+1}. {VAR_LABELS[a]:<32}"
    for j in range(i):
        row += f"  r = {r_mat[i,j]:.2f}{star(p_mat[i,j])} (p = {p_mat[i,j]:.3f})"
    print(row)
print(f"\nNote. N = {len(df)}. * p < .05. ** p < .01. *** p < .001.")

# Heatmap
fig, ax = plt.subplots(figsize=(6.5, 5.2))
sns.heatmap(corr_df, annot=True, fmt=".2f", cmap="RdBu_r", vmin=-1, vmax=1,
            square=True, linewidths=1, cbar_kws={"label": "Pearson r"}, ax=ax)
ax.set_title("Correlation Matrix Heatmap", fontweight="bold", pad=12)
plt.tight_layout()
plt.savefig("/mnt/data/correlation_heatmap.png", bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# 5. Pairwise scatter plot matrix (visual inspection of relations)
# ============================================================
sns.pairplot(df[N_VARS].rename(columns=VAR_LABELS), diag_kind="kde",
             plot_kws={"alpha": .7, "s": 25, "edgecolor": "white"})
plt.savefig("/mnt/data/pairplot.png", bbox_inches="tight")
plt.show()

print("\nNotebook 01 complete. Figures saved: qq_plots.png, correlation_heatmap.png, pairplot.png")